# Debug List Scraper Kominfo Malang Kabupaten
List-only. URL debug: `https://kominfo.malangkab.go.id/berita`. Tidak scrape artikel detail.


In [1]:
from pathlib import Path
import sys
import importlib
import inspect

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / 'scrapers').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import pandas as pd
pd.set_option('display.max_colwidth', 180)
pd.set_option('display.max_rows', 200)

from scrapers.common import cutoff_date, fetch_html, fetch_text, parse_date, normalize_date, is_older_than_cutoff, records_to_df

cutoff = cutoff_date()
print('project:', PROJECT_ROOT)
print('cutoff:', cutoff.date())

import scrapers.kominfomalangkab as scraper
scraper = importlib.reload(scraper)
print('module file:', scraper.__file__)

from datetime import datetime


project: /Users/tsar/Library/Mobile Documents/com~apple~CloudDocs/main/college/pkl/project
cutoff: 2026-02-28
module file: /Users/tsar/Library/Mobile Documents/com~apple~CloudDocs/main/college/pkl/project/scrapers/kominfomalangkab.py


In [2]:
MAX_PAGES = 200
MAX_CLICKS = 40
TITLE_LIMIT = 90


def short_title(value, limit=TITLE_LIMIT):
    text = '' if pd.isna(value) else str(value).strip()
    return text if len(text) <= limit else text[: limit - 3] + '...'


def ensure_debug_columns(df):
    df = df.copy()
    if 'page_num' not in df.columns:
        df['page_num'] = pd.NA
    if 'published_date' not in df.columns:
        df['published_date'] = None
    df['published_dt'] = pd.to_datetime(
        df['published_date'].apply(parse_date),
        errors='coerce',
    )
    return df


def print_debug_rows(df):
    for _, row in df.iterrows():
        page = row.get('page_num')
        page_text = '---' if pd.isna(page) else f'{int(page):03d}'
        date_text = row.get('published_date')
        date_text = 'None' if pd.isna(date_text) else str(date_text)
        print(f"page={page_text} | date={date_text} | title={short_title(row.get('title'))}")


def audit_list(df):
    df = ensure_debug_columns(df)
    print('total rows:', len(df))
    if len(df) == 0:
        print('No rows found.')
        return df

    has_page = df['page_num'].notna().any()
    has_date = df['published_dt'].notna().any()

    print('first page:', df['page_num'].dropna().min() if has_page else None)
    print('last page:', df['page_num'].dropna().max() if has_page else None)
    print('newest date:', df['published_dt'].max() if has_date else None)
    print('oldest date:', df['published_dt'].min() if has_date else None)
    print('cutoff date:', cutoff)
    print('reached cutoff:', bool((df['published_dt'].dropna() < cutoff).any()) if has_date else False)
    print('null parsed dates:', int(df['published_dt'].isna().sum()))

    print('\nrows per month:')
    if has_date:
        display(
            df.assign(month=df['published_dt'].dt.to_period('M').astype(str))
            .groupby('month', dropna=False)
            .size()
            .reset_index(name='count')
        )
    else:
        print('No parseable published_date in list page.')

    print('\nrows per page:')
    display(
        df.groupby('page_num', dropna=False)
        .agg(
            rows=('url', 'count'),
            newest=('published_dt', 'max'),
            oldest=('published_dt', 'min'),
        )
        .reset_index()
        .tail(60)
    )
    return df


In [3]:
KOMINFO_BASE_URL = 'https://kominfo.malangkab.go.id/berita'


def parse_kominfo_date(value):
    if not value:
        return None

    text = str(value).strip()
    for fmt in ('%d %B %y, %H:%M', '%d %B %Y, %H:%M'):
        try:
            return datetime.strptime(text, fmt)
        except ValueError:
            pass

    return parse_date(text)


def debug_scrape_list(cutoff, max_pages=MAX_PAGES):
    rows = []
    for page_num in range(1, max_pages + 1):
        url = KOMINFO_BASE_URL if page_num == 1 else f'{KOMINFO_BASE_URL}?page={page_num}'
        print(f'Scraping Kominfo page {page_num}: {url}')
        try:
            soup = fetch_html(url, verify=False)
        except Exception as error:
            print(f'Gagal buka Kominfo page {page_num}: {error}')
            break

        cards = soup.select('div.bg-white.rounded-xl.shadow-lg')
        if not cards:
            break

        stop = False
        for card in cards:
            title_tag = card.select_one('h3')
            link_tag = card.select_one("a.block.flex-grow[href*='/berita/']") or card.select_one("a[href*='/berita/']")
            date_tag = card.select_one('p.text-sm.text-gray-500')
            if not title_tag or not link_tag:
                continue

            published_date = date_tag.get_text(' ', strip=True) if date_tag else None
            published_dt = parse_kominfo_date(published_date)

            if published_dt and published_dt < cutoff:
                stop = True
                break

            rows.append({
                'page_num': page_num,
                'list_page_url': url,
                'title': title_tag.get_text(strip=True),
                'url': link_tag['href'],
                'published_date': published_date,
                'published_dt_raw': published_dt,
            })
        if stop:
            break
    return records_to_df(rows)


list_df = debug_scrape_list(cutoff)
list_df = ensure_debug_columns(list_df)
if 'published_dt_raw' in list_df.columns:
    list_df['published_dt'] = pd.to_datetime(list_df['published_dt_raw'], errors='coerce')
print_debug_rows(list_df)


Scraping Kominfo page 1: https://kominfo.malangkab.go.id/berita
Scraping Kominfo page 2: https://kominfo.malangkab.go.id/berita?page=2
Scraping Kominfo page 3: https://kominfo.malangkab.go.id/berita?page=3
Scraping Kominfo page 4: https://kominfo.malangkab.go.id/berita?page=4
Scraping Kominfo page 5: https://kominfo.malangkab.go.id/berita?page=5
Scraping Kominfo page 6: https://kominfo.malangkab.go.id/berita?page=6
page=001 | date=23 June 26, 14:11 | title=AI Perluas Peluang Kerja Inklusif bagi Penyandang Tunanetra
page=001 | date=22 June 26, 14:09 | title=Tingkatkan Sinergi Satu Data dan Keamanan Informasi, Diskominfo dan Statistik Kabupaten...
page=001 | date=19 June 26, 13:39 | title=Diskominfo Kabupaten Malang Gelar Audit Internal Keamanan Informasi untuk Pertahankan S...
page=001 | date=17 June 26, 09:20 | title=Mengenal Konsep AI Sandwich: Cara Bijak Menggunakan AI agar Manusia Tetap Pegang Kendali
page=001 | date=17 June 26, 09:05 | title=KIM of the Week: KIM Karya Inovatif Angk

In [4]:
list_df = audit_list(list_df)


total rows: 45
first page: 1
last page: 5
newest date: None
oldest date: None
cutoff date: 2026-02-28 09:28:39.129037
reached cutoff: False
null parsed dates: 45

rows per month:
No parseable published_date in list page.

rows per page:


,page_num,rows,newest,oldest
0,1,9,NaT,NaT
1,2,9,NaT,NaT
2,3,9,NaT,NaT
3,4,9,NaT,NaT
4,5,9,NaT,NaT


In [5]:
output_path = PROJECT_ROOT / 'csv' / 'kominfomalangkab_list_debug.csv'
list_df.to_csv(output_path, index=False)
output_path


PosixPath('/Users/tsar/Library/Mobile Documents/com~apple~CloudDocs/main/college/pkl/project/csv/kominfomalangkab_list_debug.csv')

## Optional article sample check

Ambil beberapa artikel detail, cek `content`, lalu simpan ke article debug CSV.

In [ ]:

ARTICLE_SAMPLE_SIZE = 5
ARTICLE_DEBUG_OUTPUT_PATH = PROJECT_ROOT / 'csv' / 'kominfo_malangkab_article_debug.csv'

article_rows = []
sample_rows = list_df.head(ARTICLE_SAMPLE_SIZE)
for index, row in sample_rows.iterrows():
    try:
        article = scraper.extract_article(row)
        article_rows.append(article)
        print(f"[{len(article_rows)}] content_len={len(article.get('content') or '')} | {short_title(article.get('title'))}")
    except Exception as error:
        print(f"failed sample article {index + 1}: {row.get('url')} | {error}")

article_df = pd.DataFrame(article_rows)
ARTICLE_DEBUG_OUTPUT_PATH.parent.mkdir(exist_ok=True)
article_df.to_csv(ARTICLE_DEBUG_OUTPUT_PATH, index=False)
print('saved:', ARTICLE_DEBUG_OUTPUT_PATH)
display(article_df[['title', 'published_date', 'content', 'url']].head() if len(article_df) else article_df)
